# Qwen3.5-4B 한글 텍스트 요약/번역 파인튜닝



Unsloth + LoRA를 사용하여 **Qwen3.5-4B (Vision 모델)**의 언어 레이어만 파인튜닝하는 노트북입니다.


**모델**: `unsloth/Qwen3.5-4B` (비전 모델, 언어 레이어만 학습)
**태스크**: 한글 텍스트 요약 및 번역
**데이터**: Google Drive의 커스텀 JSONL 데이터셋

Google Colab **무료 Tesla T4** 인스턴스에서 실행 가능합니다.

author: kwonkj(kwonkj@kfirstlab.com)

### Google Drive 마운트

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


### 패키지 설치

In [3]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0

### 모델 로드

In [4]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-4B",
    load_in_4bit = True,                    # Colab T4 메모리 절약
    use_gradient_checkpointing = "unsloth",  # 긴 컨텍스트 지원
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/336 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

### LoRA 어댑터 설정

비전 레이어는 학습하지 않고 **언어 레이어만** 효율적으로 학습합니다.

In [5]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # 비전 레이어 학습 안 함
    finetune_language_layers   = True,   # 언어 레이어만 학습
    finetune_attention_modules = True,   # 어텐션 모듈 학습
    finetune_mlp_modules       = True,   # MLP 모듈 학습
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth: Making `model.base_model.model.model.language_model` require gradients


### 데이터 준비

Google Drive에서 JSONL 데이터셋을 로드합니다.

### (선택) 테스트용 샘플 데이터 생성

실제 JSONL 파일이 없을 때 파이프라인 테스트용 가짜 데이터를 생성합니다.  
실제 데이터가 있으면 이 셀을 **건너뛰세요**.

In [6]:
# ========================================
# 테스트용 가짜 데이터 생성 (파이프라인 검증용)
# 실제 데이터가 있으면 이 셀을 건너뛰세요.
# ========================================
import json, os

sample_data = [
    # 요약 태스크
    {
        "messages": [
            {"role": "user", "content": "다음 글을 요약해줘:\n\n인공지능(AI)은 컴퓨터 시스템이 인간의 지능을 모방하여 학습, 추론, 자기 교정 등의 작업을 수행할 수 있도록 하는 기술입니다. 최근 딥러닝과 대규모 언어 모델의 발전으로 AI는 자연어 처리, 이미지 인식, 자율주행 등 다양한 분야에서 획기적인 성과를 보이고 있습니다. 특히 GPT, BERT 등의 트랜스포머 기반 모델은 텍스트 생성과 이해 능력에서 인간 수준에 근접한 성능을 달성했습니다."},
            {"role": "assistant", "content": "인공지능은 인간 지능을 모방하는 기술로, 딥러닝과 대규모 언어 모델의 발전으로 자연어 처리, 이미지 인식 등에서 큰 성과를 보이고 있으며, 트랜스포머 기반 모델이 텍스트 생성에서 인간 수준의 성능을 달성했습니다."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "다음 글을 요약해줘:\n\n한국은 사계절이 뚜렷한 나라로, 봄에는 벚꽃이 만개하고 여름에는 무더위와 장마가 찾아옵니다. 가을에는 단풍이 아름답게 물들고 겨울에는 눈이 내려 스키와 같은 겨울 스포츠를 즐길 수 있습니다. 이러한 사계절의 변화는 한국의 문화와 음식에도 깊은 영향을 미쳐, 계절마다 고유한 축제와 제철 음식을 즐기는 전통이 있습니다."},
            {"role": "assistant", "content": "한국은 사계절이 뚜렷하여 각 계절마다 고유한 자연 경관과 축제, 제철 음식 등의 문화적 전통을 가지고 있습니다."}
        ]
    },
    # 번역 태스크
    {
        "messages": [
            {"role": "user", "content": "다음 문장을 영어로 번역해줘:\n\n오늘 회의에서 논의된 주요 안건은 다음 분기 예산 배분과 신규 프로젝트 일정 조율이었습니다."},
            {"role": "assistant", "content": "The main agenda items discussed at today's meeting were next quarter's budget allocation and new project schedule coordination."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "다음 문장을 영어로 번역해줘:\n\n머신러닝 모델의 성능을 향상시키기 위해서는 양질의 데이터 확보와 적절한 하이퍼파라미터 튜닝이 필수적입니다."},
            {"role": "assistant", "content": "To improve the performance of machine learning models, securing high-quality data and proper hyperparameter tuning are essential."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "다음 글을 요약해줘:\n\n클라우드 컴퓨팅은 인터넷을 통해 서버, 스토리지, 데이터베이스, 네트워킹, 소프트웨어 등의 컴퓨팅 서비스를 제공하는 기술입니다. 기업들은 클라우드를 통해 초기 인프라 투자 비용을 절감하고, 필요에 따라 자원을 유연하게 확장하거나 축소할 수 있습니다. AWS, Azure, GCP 등이 대표적인 클라우드 서비스 제공업체입니다."},
            {"role": "assistant", "content": "클라우드 컴퓨팅은 인터넷으로 컴퓨팅 서비스를 제공하는 기술로, 기업이 인프라 비용을 절감하고 자원을 유연하게 관리할 수 있게 해주며, AWS, Azure, GCP가 대표적입니다."}
        ]
    },
]

# JSONL 파일로 저장
sample_path = "/content/sample_test_data.jsonl"
with open(sample_path, "w", encoding="utf-8") as f:
    for item in sample_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"테스트 샘플 {len(sample_data)}개 생성 완료: {sample_path}")
print(f"태스크 구성: 요약 3개, 번역 2개")
print()
print("첫 번째 샘플 미리보기:")
print(json.dumps(sample_data[0], ensure_ascii=False, indent=2))

테스트 샘플 5개 생성 완료: /content/sample_test_data.jsonl
태스크 구성: 요약 3개, 번역 2개

첫 번째 샘플 미리보기:
{
  "messages": [
    {
      "role": "user",
      "content": "다음 글을 요약해줘:\n\n인공지능(AI)은 컴퓨터 시스템이 인간의 지능을 모방하여 학습, 추론, 자기 교정 등의 작업을 수행할 수 있도록 하는 기술입니다. 최근 딥러닝과 대규모 언어 모델의 발전으로 AI는 자연어 처리, 이미지 인식, 자율주행 등 다양한 분야에서 획기적인 성과를 보이고 있습니다. 특히 GPT, BERT 등의 트랜스포머 기반 모델은 텍스트 생성과 이해 능력에서 인간 수준에 근접한 성능을 달성했습니다."
    },
    {
      "role": "assistant",
      "content": "인공지능은 인간 지능을 모방하는 기술로, 딥러닝과 대규모 언어 모델의 발전으로 자연어 처리, 이미지 인식 등에서 큰 성과를 보이고 있으며, 트랜스포머 기반 모델이 텍스트 생성에서 인간 수준의 성능을 달성했습니다."
    }
  ]
}


In [7]:
import json

# 실제 데이터 경로 또는 위에서 생성한 샘플 데이터 경로를 선택하세요
# data_path = "/content/drive/MyDrive/test300.jsonl"  # 실제 데이터
data_path = "/content/sample_test_data.jsonl"     # 테스트용 샘플 데이터

raw_data = []
with open(data_path, "r", encoding="utf-8") as f:
    for line in f:
        raw_data.append(json.loads(line))

print(f"총 로드된 샘플 수: {len(raw_data)}")

총 로드된 샘플 수: 5


### 데이터 구조 확인

변환 전에 원본 데이터의 구조를 확인합니다.

In [8]:
# 첫 번째 샘플의 구조 확인

print(json.dumps(raw_data[0], ensure_ascii=False, indent=2))

{
  "messages": [
    {
      "role": "user",
      "content": "다음 글을 요약해줘:\n\n인공지능(AI)은 컴퓨터 시스템이 인간의 지능을 모방하여 학습, 추론, 자기 교정 등의 작업을 수행할 수 있도록 하는 기술입니다. 최근 딥러닝과 대규모 언어 모델의 발전으로 AI는 자연어 처리, 이미지 인식, 자율주행 등 다양한 분야에서 획기적인 성과를 보이고 있습니다. 특히 GPT, BERT 등의 트랜스포머 기반 모델은 텍스트 생성과 이해 능력에서 인간 수준에 근접한 성능을 달성했습니다."
    },
    {
      "role": "assistant",
      "content": "인공지능은 인간 지능을 모방하는 기술로, 딥러닝과 대규모 언어 모델의 발전으로 자연어 처리, 이미지 인식 등에서 큰 성과를 보이고 있으며, 트랜스포머 기반 모델이 텍스트 생성에서 인간 수준의 성능을 달성했습니다."
    }
  ]
}


### 학습 데이터 전처리

`messages`를 `UnslothVisionDataCollator`가 처리할 수 있는 대화 형식으로 변환합니다.
이미지 없이 텍스트만 사용하는 경우에도 Vision Data Collator를 사용해야 합니다.

In [9]:
def convert_to_conversation(sample):
    """JSONL 샘플을 비전 모델 대화 형식으로 변환 (이미지 없이 텍스트만)"""
    user_content = sample["messages"][0]["content"]
    assistant_content = sample["messages"][1]["content"]
    return {
        "messages": [
            {
                "role": "user",
                "content": [{"type": "text", "text": user_content}]
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": assistant_content}]
            },
        ]
    }

converted_dataset = [convert_to_conversation(sample) for sample in raw_data]

print(f"변환 완료: {len(converted_dataset)}개 샘플")
print()
print("첫 번째 샘플 미리보기:")
import json as _json
print(_json.dumps(converted_dataset[0], ensure_ascii=False, indent=2))

변환 완료: 5개 샘플

첫 번째 샘플 미리보기:
{
  "messages": [
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": "다음 글을 요약해줘:\n\n인공지능(AI)은 컴퓨터 시스템이 인간의 지능을 모방하여 학습, 추론, 자기 교정 등의 작업을 수행할 수 있도록 하는 기술입니다. 최근 딥러닝과 대규모 언어 모델의 발전으로 AI는 자연어 처리, 이미지 인식, 자율주행 등 다양한 분야에서 획기적인 성과를 보이고 있습니다. 특히 GPT, BERT 등의 트랜스포머 기반 모델은 텍스트 생성과 이해 능력에서 인간 수준에 근접한 성능을 달성했습니다."
        }
      ]
    },
    {
      "role": "assistant",
      "content": [
        {
          "type": "text",
          "text": "인공지능은 인간 지능을 모방하는 기술로, 딥러닝과 대규모 언어 모델의 발전으로 자연어 처리, 이미지 인식 등에서 큰 성과를 보이고 있으며, 트랜스포머 기반 모델이 텍스트 생성에서 인간 수준의 성능을 달성했습니다."
        }
      ]
    }
  ]
}


### 파인튜닝 전 추론 테스트

파인튜닝 전 모델의 기본 출력을 확인합니다.

In [10]:
FastVisionModel.for_inference(model)

# 학습 데이터의 첫 번째 샘플로 테스트
test_input = raw_data[0]["messages"][0]["content"]
print("=== 입력 텍스트 ===")
print(test_input[:300], "..." if len(test_input) > 300 else "")
print()

messages = [{"role": "user", "content": [{"type": "text", "text": test_input}]}]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(
    None,
    input_text,
    add_special_tokens=False,
    return_tensors="pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print("=== 파인튜닝 전 출력 ===")
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=256,
    use_cache=True,
    temperature=0.7,
    min_p=0.1,
)

=== 입력 텍스트 ===
다음 글을 요약해줘:

인공지능(AI)은 컴퓨터 시스템이 인간의 지능을 모방하여 학습, 추론, 자기 교정 등의 작업을 수행할 수 있도록 하는 기술입니다. 최근 딥러닝과 대규모 언어 모델의 발전으로 AI는 자연어 처리, 이미지 인식, 자율주행 등 다양한 분야에서 획기적인 성과를 보이고 있습니다. 특히 GPT, BERT 등의 트랜스포머 기반 모델은 텍스트 생성과 이해 능력에서 인간 수준에 근접한 성능을 달성했습니다. 

=== 파인튜닝 전 출력 ===
인공지능 (AI) 은 인간의 지능을 모방하여 학습과 추론을 수행하는 기술로, 최근 딥러닝과 대규모 언어 모델의 발전으로 자연어 처리, 이미지 인식, 자율주행 등 다양한 분야에서 혁신적인 성과를 거두고 있습니다. 특히 GPT, BERT 등의 트랜스포머 기반 모델은 텍스트 생성 및 이해 능력에서 인간 수준에 근접한 성능을 보여주고 있습니다.<|im_end|>
<|endoftext|>


<a name="Train"></a>
### 모델 학습

`max_steps=30`은 빠른 테스트용입니다. 전체 학습시 `num_train_epochs=3`으로 변경하세요.

In [11]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=converted_dataset,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=-1,
        num_train_epochs=3,  # 전체 학습 시 max_steps 대신 이것을 사용
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        # 비전 모델 파인튜닝 필수 설정:
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=2048,
    ),
)

Unsloth: Model does not have a default image size - using 512


In [12]:
# GPU 메모리 확인

gpu_stats = torch.cuda.get_device_properties(0)

start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)

max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")

print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-40GB. Max memory = 39.494 GB.
8.879 GB of memory reserved.


In [13]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5 | Num Epochs = 3 | Total steps = 3
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 32,464,896 of 4,571,730,432 (0.71% trained)


Step,Training Loss
1,1.464806
2,1.464449
3,1.380422


In [14]:
# 학습 결과 통계

used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)

used_memory_for_lora = round(used_memory - start_gpu_memory, 3)

used_percentage = round(used_memory / max_memory * 100, 3)

lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")

print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")

print(f"Peak reserved memory = {used_memory} GB.")

print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")

print(f"Peak reserved memory % of max memory = {used_percentage} %.")

print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

125.4564 seconds used for training.
2.09 minutes used for training.
Peak reserved memory = 9.109 GB.
Peak reserved memory for training = 0.23 GB.
Peak reserved memory % of max memory = 23.064 %.
Peak reserved memory for training % of max memory = 0.582 %.


<a name="Inference"></a>
### 파인튜닝 후 추론 테스트

동일한 입력으로 파인튜닝 전후 결과를 비교합니다.

In [15]:
FastVisionModel.for_inference(model)

# 학습 데이터의 첫 번째 샘플로 테스트
test_input = raw_data[0]["messages"][0]["content"]
print("=== 입력 텍스트 ===")
print(test_input[:300], "..." if len(test_input) > 300 else "")
print()

messages = [{"role": "user", "content": [{"type": "text", "text": test_input}]}]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(
    None,
    input_text,
    add_special_tokens=False,
    return_tensors="pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print("=== 파인튜닝 후 출력 ===")
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=256,
    use_cache=True,
    temperature=0.7,
    min_p=0.1,
)

=== 입력 텍스트 ===
다음 글을 요약해줘:

인공지능(AI)은 컴퓨터 시스템이 인간의 지능을 모방하여 학습, 추론, 자기 교정 등의 작업을 수행할 수 있도록 하는 기술입니다. 최근 딥러닝과 대규모 언어 모델의 발전으로 AI는 자연어 처리, 이미지 인식, 자율주행 등 다양한 분야에서 획기적인 성과를 보이고 있습니다. 특히 GPT, BERT 등의 트랜스포머 기반 모델은 텍스트 생성과 이해 능력에서 인간 수준에 근접한 성능을 달성했습니다. 

=== 파인튜닝 후 출력 ===
인공지능 (AI) 은 인간의 지능을 모방하여 학습과 추론을 수행하는 기술로, 최근 딥러닝과 대규모 언어 모델의 발전으로 자연어 처리, 이미지 인식, 자율주행 등 다양한 분야에서 획기적인 성과를 내고 있습니다. 특히 트랜스포머 기반 모델은 텍스트 생성 및 이해 능력에서 인간 수준에 근접한 성능을 달성했습니다.<|im_end|>


다른 샘플로도 테스트해봅니다.

In [16]:
# 두 번째 샘플로 테스트
if len(raw_data) > 1:
    test_input_2 = raw_data[1]["messages"][0]["content"]
    print("=== 입력 텍스트 (샘플 2) ===")
    print(test_input_2[:300], "..." if len(test_input_2) > 300 else "")
    print()
    messages_2 = [{"role": "user", "content": [{"type": "text", "text": test_input_2}]}]
    input_text_2 = tokenizer.apply_chat_template(messages_2, add_generation_prompt=True)
    inputs_2 = tokenizer(
        None,
        input_text_2,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")
    print("=== 파인튜닝 후 출력 (샘플 2) ===")
    _ = model.generate(
        **inputs_2,
        streamer=text_streamer,
        max_new_tokens=256,
        use_cache=True,
        temperature=0.7,
        min_p=0.1,
    )

=== 입력 텍스트 (샘플 2) ===
다음 글을 요약해줘:

한국은 사계절이 뚜렷한 나라로, 봄에는 벚꽃이 만개하고 여름에는 무더위와 장마가 찾아옵니다. 가을에는 단풍이 아름답게 물들고 겨울에는 눈이 내려 스키와 같은 겨울 스포츠를 즐길 수 있습니다. 이러한 사계절의 변화는 한국의 문화와 음식에도 깊은 영향을 미쳐, 계절마다 고유한 축제와 제철 음식을 즐기는 전통이 있습니다. 

=== 파인튜닝 후 출력 (샘플 2) ===
한국은 뚜렷한 사계절로 인해 계절마다 벚꽃, 장마, 단풍, 눈 등 다양한 자연 풍경을 즐길 수 있습니다. 이러한 계절의 변화는 한국의 문화와 음식에 깊이 반영되어, 각 계절마다 고유한 축제와 제철 음식을 즐기는 전통이 자리 잡고 있습니다.<|im_end|>


번역 태스크도 테스트해봅니다.

In [17]:
# 번역 태스크 테스트 (데이터에 번역 샘플이 있는 경우)
translation_idx = None
for idx, sample in enumerate(raw_data):
    content = sample["messages"][0]["content"]
    if "번역" in content or "translate" in content.lower():
        translation_idx = idx
        break

if translation_idx is not None:
    test_input_t = raw_data[translation_idx]["messages"][0]["content"]
    print(f"=== 번역 입력 (샘플 {translation_idx}) ===")
    print(test_input_t[:500], "..." if len(test_input_t) > 500 else "")
    print()
    messages_t = [{"role": "user", "content": [{"type": "text", "text": test_input_t}]}]
    input_text_t = tokenizer.apply_chat_template(messages_t, add_generation_prompt=True)
    inputs_t = tokenizer(
        None,
        input_text_t,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")
    print("=== 번역 출력 ===")
    _ = model.generate(
        **inputs_t,
        streamer=text_streamer,
        max_new_tokens=256,
        use_cache=True,
        temperature=0.7,
        min_p=0.1,
    )
    print()
    print("=== 번역 정답 ===")
    print(raw_data[translation_idx]["messages"][1]["content"])
else:
    print("번역 태스크 샘플을 찾을 수 없습니다.")

=== 번역 입력 (샘플 2) ===
다음 문장을 영어로 번역해줘:

오늘 회의에서 논의된 주요 안건은 다음 분기 예산 배분과 신규 프로젝트 일정 조율이었습니다. 

=== 번역 출력 ===
The main agenda items discussed in today's meeting were the budget allocation for the next quarter and the scheduling of new projects.<|im_end|>

=== 번역 정답 ===
The main agenda items discussed at today's meeting were next quarter's budget allocation and new project schedule coordination.


### 정답과 비교

In [18]:
# 정답(ground truth)과 비교

print("=== 정답 (샘플 1) ===")

print(raw_data[0]["messages"][1]["content"])

print()

if len(raw_data) > 1:
    print("=== 정답 (샘플 2) ===")
    print(raw_data[1]["messages"][1]["content"])

=== 정답 (샘플 1) ===
인공지능은 인간 지능을 모방하는 기술로, 딥러닝과 대규모 언어 모델의 발전으로 자연어 처리, 이미지 인식 등에서 큰 성과를 보이고 있으며, 트랜스포머 기반 모델이 텍스트 생성에서 인간 수준의 성능을 달성했습니다.

=== 정답 (샘플 2) ===
한국은 사계절이 뚜렷하여 각 계절마다 고유한 자연 경관과 축제, 제철 음식 등의 문화적 전통을 가지고 있습니다.


<a name="Save"></a>
### 모델 저장

LoRA 어댑터를 로컬 및 Hugging Face에 저장합니다.

**참고**: 아래는 LoRA 어댑터만 저장합니다. 전체 모델 저장은 하단의 float16/GGUF 섹션을 참고하세요.

In [19]:
# model.save_pretrained("/content/drive/로컬저장할폴더이름")  # 로컬 저장

# tokenizer.save_pretrained("qwen_lora")

# model.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN")  # HF 업로드

# tokenizer.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN")

저장된 LoRA 어댑터를 다시 로드하여 추론하려면 아래 `False`를 `True`로 변경하세요.

In [20]:
#if False:
#    from unsloth import FastVisionModel
#    model, tokenizer = FastVisionModel.from_pretrained(
#        model_name="qwen_lora",
#        load_in_4bit=True,
#    )
#    FastVisionModel.for_inference(model)
#    test_input = raw_data[0]["messages"][0]["content"]
#    messages = [{"role": "user", "content": [{"type": "text", "text": test_input}]}]
#    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
#    inputs = tokenizer(
#       None, input_text, add_special_tokens=False,
#        return_tensors="pt",
#    ).to("cuda")
#    from transformers import TextStreamer
#    text_streamer = TextStreamer(tokenizer, skip_prompt=True)
#    _ = model.generate(
#        **inputs, streamer=text_streamer, max_new_tokens=256,
#        use_cache=True, temperature=0.7, min_p=0.1,
#    )

### float16으로 저장 (VLLM용)

`float16`으로 머지하여 저장합니다. VLLM 서빙에 적합합니다.

In [21]:
# 로컬에 16bit로 저장

#if False: model.save_pretrained_merged("unsloth_finetune", tokenizer)



# Hugging Face에 업로드

#if False: model.push_to_hub_merged("YOUR_USERNAME/unsloth_finetune", tokenizer, token="YOUR_HF_TOKEN")

### GGUF / llama.cpp 변환

GGUF 형식으로 저장합니다. `q8_0`이 기본이며, `q4_k_m` 등 다양한 양자화를 지원합니다.

In [22]:
# Q8_0으로 저장

#if False: model.save_pretrained_gguf("qwen_finetune", tokenizer)

#if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, token="YOUR_HF_TOKEN")



# f16으로 저장

#if False: model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method="f16")

#if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method="f16", token="YOUR_HF_TOKEN")



# q4_k_m으로 저장

#if False: model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method="q4_k_m")

#if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method="q4_k_m", token="YOUR_HF_TOKEN")



# 여러 양자화 한번에 저장

#if False:
#    model.push_to_hub_gguf(
#        "HF_USERNAME/qwen_finetune",
#        tokenizer,
#        quantization_method=["q4_k_m", "q8_0", "q5_k_m"],
#        token="YOUR_HF_TOKEN",
#    )

In [23]:
import os
from google.colab import runtime

# 모든 작업이 끝나면 런타임을 종료합니다.
runtime.unassign()

### 완료!

Qwen3.5-4B 한글 텍스트 요약/번역 파인튜닝이 완료되었습니다.

**설정 요약**:
- `FastVisionModel` + `unsloth/Qwen3.5-4B` (비전 모델)
- `finetune_vision_layers=False` (언어 레이어만 학습)
- `UnslothVisionDataCollator` 사용 (비전 모델 필수)
- `load_in_4bit=True` (Colab T4 메모리 최적화)
- 텍스트 전용 데이터로 학습 (이미지 미사용)

**추가 리소스**:
- [Unsloth Discord](https://discord.gg/unsloth)
- [Unsloth 문서](https://docs.unsloth.ai)
- [Unsloth GitHub](https://github.com/unslothai/unsloth)

## 비교: Qwen3.5-4B(Vision) model vs Qwen3-4B Text only model

Qwen3.5-4B(Vision)로 텍스트만 파인튜닝 — 가능하지만 비추천
가능은 합니다
Unsloth에서 finetune_vision_layers=False, finetune_language_layers=True로 설정하면 비전 레이어는 건드리지 않고 언어 레이어만 학습할 수 있습니다.

하지만 현재 설정(Qwen3-4B)을 유지하는 게 낫습니다
비교 항목	Qwen3.5-4B (Vision)	Qwen3-4B (텍스트)
로드 방식	FastModel (통합 클래스)	FastLanguageModel (간단)
VRAM	비전 인코더도 로드 → 추가 메모리 소비	텍스트만 → 효율적
설정 복잡도	processor.tokenizer 추출, 커스텀 Data Collator 필요	표준 SFTTrainer 그대로 사용
알려진 이슈	images=None 명시 필요 등 (GitHub #1436, #1590)	안정적
텍스트 성능	Qwen3 대비 소폭 향상 가능	충분히 우수
핵심 이유
Qwen3.5-4B를 FastLanguageModel로 로드 불가 — 비전 인코더 때문에 FastModel을 써야 하고, 이 경우 tokenizer 대신 processor가 반환되어 추가 처리가 필요
비전 인코더가 VRAM을 잡아먹음 — Colab T4(16GB)에서 불필요한 메모리 낭비
텍스트 요약/번역에 비전 레이어는 완전히 불필요 — 복잡성만 증가
권장
한글 텍스트 요약/번역만 목적이라면 현재 Qwen3-4B 텍스트 전용 설정이 최적입니다.
Qwen3.5의 소폭 성능 향상이 설정 복잡도와 메모리 오버헤드를 감수할 만큼 크지 않습니다.

나중에 이미지+텍스트 멀티모달 파인튜닝이 필요해지면 그때 Qwen3.5-4B Vision으로 전환하면 됩니다.